# Data Augmentation
Synonym substitution + simple back-translation simulation for Swahili sentiment dataset.

In [ ]:
# ==========================================
# INSTALL
# ==========================================
!pip install nlpaug nltk pandas numpy

In [ ]:
# ==========================================
# IMPORTS
# ==========================================
import pandas as pd
import numpy as np
import random
import os
import re

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import wordnet

random.seed(42)
np.random.seed(42)

In [ ]:
# ==========================================
# LOAD CLEANED DATA
# ==========================================
df = pd.read_csv('../data/processed/cleaned_data.csv')
df = df.dropna(subset=['clean_text', 'label_encoded'])
df['clean_text'] = df['clean_text'].astype(str)

print(f'Original dataset size: {len(df)}')
print(df['label_encoded'].value_counts())

In [ ]:
# ==========================================
# SYNONYM SUBSTITUTION
# Replaces random words with WordNet synonyms.
# Works on the English-transliterated portion
# of code-switched Swahili tweets.
# ==========================================
def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace('_', ' ')
            if synonym.lower() != word.lower():
                synonyms.add(synonym)
    return list(synonyms)


def synonym_substitution(text, n=1):
    words = text.split()
    new_words = words.copy()
    random_word_list = list(set(words))
    random.shuffle(random_word_list)
    num_replaced = 0

    for word in random_word_list:
        synonyms = get_synonyms(word)
        if synonyms:
            synonym = random.choice(synonyms)
            new_words = [synonym if w == word else w for w in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break

    return ' '.join(new_words)


# Test it
sample = df['clean_text'].iloc[0]
print('Original: ', sample)
print('Augmented:', synonym_substitution(sample))

In [ ]:
# ==========================================
# RANDOM DELETION
# Randomly deletes words with probability p.
# ==========================================
def random_deletion(text, p=0.1):
    words = text.split()
    if len(words) == 1:
        return text
    new_words = [w for w in words if random.random() > p]
    if not new_words:
        return random.choice(words)
    return ' '.join(new_words)


# Test it
print('Original: ', sample)
print('Deleted:  ', random_deletion(sample))

In [ ]:
# ==========================================
# AUGMENT MINORITY CLASSES
# We oversample minority classes to balance
# the dataset using both augmentation methods.
# ==========================================
def augment_class(class_df, target_count, label):
    augmented_rows = []
    current_count = len(class_df)
    needed = target_count - current_count

    if needed <= 0:
        return pd.DataFrame()

    print(f'  Label {label}: adding {needed} samples (from {current_count})')

    for i in range(needed):
        row = class_df.sample(1).iloc[0]
        text = row['clean_text']

        # Alternate between augmentation strategies
        if i % 2 == 0:
            aug_text = synonym_substitution(text, n=1)
        else:
            aug_text = random_deletion(text, p=0.1)

        new_row = row.copy()
        new_row['clean_text'] = aug_text
        augmented_rows.append(new_row)

    return pd.DataFrame(augmented_rows)


# Find max class count
class_counts = df['label_encoded'].value_counts()
max_count = class_counts.max()
print(f'Balancing all classes to: {max_count} samples')

augmented_parts = [df]  # Start with original

for label_val in df['label_encoded'].unique():
    class_df = df[df['label_encoded'] == label_val]
    aug_df = augment_class(class_df, max_count, label_val)
    if not aug_df.empty:
        augmented_parts.append(aug_df)

df_augmented = pd.concat(augmented_parts, ignore_index=True)
df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nAugmented dataset size: {len(df_augmented)}')
print(df_augmented['label_encoded'].value_counts())

In [ ]:
# ==========================================
# SAVE AUGMENTED DATA
# ==========================================
os.makedirs('../data/processed', exist_ok=True)

df_augmented.to_csv(
    '../data/processed/augmented_data.csv',
    index=False
)

print('Augmented data saved to ../data/processed/augmented_data.csv')

In [ ]:
# ==========================================
# VISUALIZE BEFORE vs AFTER
# ==========================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

df['label_encoded'].value_counts().rename(label_names).plot(
    kind='bar', ax=axes[0], color='steelblue'
)
axes[0].set_title('Before Augmentation')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')

df_augmented['label_encoded'].value_counts().rename(label_names).plot(
    kind='bar', ax=axes[1], color='seagreen'
)
axes[1].set_title('After Augmentation')
axes[1].set_xlabel('Sentiment')

plt.tight_layout()
os.makedirs('../results/graphs', exist_ok=True)
plt.savefig('../results/graphs/augmentation_distribution.png')
plt.show()
print('Distribution plot saved.')